# 01 · Exploratory data analysis

Run this after putting the CICIDS2017 CSVs in `data/raw/` (or the synthetic sample). Record your findings in the markdown cells. They go straight into the report.

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from nids.config import load_config, resolve
from nids.data.load import load_raw
from nids.data.clean import clean, map_label, LEAKY_COLUMNS
cfg = load_config('../configs/train.yaml')
raw = load_raw(resolve(cfg['data']['raw_dir']), sample_frac=cfg['data'].get('sample_frac', 1.0))
raw.columns = raw.columns.str.strip()
raw.shape

## 1. Raw labels and class balance

In [ ]:
counts = raw['Label'].value_counts()
display(counts.to_frame('flows').assign(share=lambda d: (d.flows / d.flows.sum()).round(4)))
raw['Label'].map(map_label).value_counts().plot.barh(logx=True, title='Mapped classes (log scale)');

**Finding:** _How imbalanced is it? Which classes are tiny?_

## 2. Data quality: inf, NaN, duplicates, constant columns

In [ ]:
num = raw.select_dtypes('number')
quality = pd.DataFrame({'inf': np.isinf(num).sum(), 'nan': num.isna().sum()})
display(quality[(quality.inf > 0) | (quality.nan > 0)])
print('duplicate rows:', raw.duplicated().sum(), f'({raw.duplicated().mean():.1%})')
print('constant columns:', [c for c in num if num[c].nunique() <= 1])
print('leaky identifier columns present:', [c for c in raw.columns if c in LEAKY_COLUMNS])

## 3. Cleaning log (row counts for the report)

In [ ]:
df = clean(raw, drop_other=cfg['data'].get('drop_other', True))
print(f'raw rows: {len(raw):,}  ->  clean rows: {len(df):,}  ({1 - len(df)/len(raw):.1%} removed)')
df['Label'].value_counts()

**Finding:** _Which step removed the most rows, and why does that matter?_

## 4. Feature correlation

In [ ]:
X = df.drop(columns='Label')
corr = X.corr().abs()
plt.figure(figsize=(10, 8)); plt.imshow(corr, cmap='viridis'); plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6)
plt.yticks(range(len(corr)), corr.columns, fontsize=6); plt.title('|correlation|'); plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().sort_values(ascending=False)
pairs.head(15)

**Finding:** _Which features are near-duplicates (|r| > 0.95)? Feature selection should remove most of them._

## 5. Key features per class

In [ ]:
key = [c for c in ['Flow Packets/s', 'Flow Duration', 'Average Packet Size', 'Total Fwd Packets', 'Destination Port'] if c in df]
fig, axes = plt.subplots(1, len(key), figsize=(4 * len(key), 4))
for ax, f in zip(np.atleast_1d(axes), key):
    df.boxplot(column=f, by='Label', ax=ax, rot=60, showfliers=False)
    ax.set_yscale('symlog'); ax.set_title(f)
plt.suptitle(''); plt.tight_layout(); plt.show()

## 6. Should `Destination Port` be a feature?

In [ ]:
df.groupby('Label')['Destination Port'].agg(lambda s: s.value_counts().head(3).to_dict())

**Decision:** _If an attack class is almost always one port (e.g. BruteForce → 22/21), the model may just learn the port. Train once with and once without this column (drop it in `clean.py`) and compare per-class recall. Write down which you chose and why._